# 🎯 Price Optimization & Scenario Analysis
In this notebook, we use our trained demand forecasting model to find the optimal price that maximizes profit.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils.config import *
from src.models.demand_forecaster import DemandForecaster
from src.optimization.price_optimizer import PriceOptimizer
from src.optimization.scenario_engine import ScenarioEngine
from src.visualization.plots import plot_profit_curve, plot_demand_curve, plot_scenario_comparison

## 1. Load Trained Model
Load the demand forecasting model we trained previously.

In [ ]:
demand_model = DemandForecaster()
demand_model.load(MODELS_DIR / 'lgbm_demand_model.pkl')
print("Model loaded successfully.")

## 2. The Optimization Problem
We aim to maximize profit $\Pi(p)$ for a given price $p$:

$$ \Pi(p) = (p - c) \times D(p) $$

Where:
- $p$ is the selling price
- $c$ is the unit cost
- $D(p)$ is the predicted demand (units sold) at price $p$

Since $D(p)$ generally decreases as $p$ increases, there is an optimal price that balances margin per unit and total volume.

## 3. Optimize Single Product
Let's optimize the price for PROD_001 under typical conditions.

In [ ]:
prod_id = 'PROD_001'
prod_info = PRODUCT_CATALOG[prod_id]
cost_price = prod_info['cost_price']
min_price = cost_price * 1.05  # At least 5% margin
max_price = cost_price * 3.0   # Cap at 3x cost

optimizer = PriceOptimizer(demand_model, cost_price, min_price, max_price)

# Construct base features for a typical day
base_features = pd.DataFrame([{
    'product_id': prod_id,
    'day_of_week': 2, # Wednesday
    'is_weekend': 0,
    'is_festival': 0,
    'discount_pct': 0,
    'rating': 4.5,
    'ad_spend': 100
}])

competitor_price = prod_info['base_price'] * 0.95 # Competitor undercutting by 5%

opt_result = optimizer.optimize(base_features, competitor_price)
print(f"Optimal Price: ${opt_result['optimal_price']:.2f}")
print(f"Predicted Demand: {opt_result['predicted_demand']:.1f} units")
print(f"Expected Daily Profit: ${opt_result['daily_profit']:.2f}")

## 4. Profit Curve Visualization
Visualize how profit varies with price to see the optimal point.

In [ ]:
curve_data = optimizer.generate_profit_curve(base_features, competitor_price)

fig_profit = plot_profit_curve(
    curve_data['prices'], 
    curve_data['profits'], 
    opt_result['optimal_price'], 
    opt_result['daily_profit'], 
    current_price=prod_info['base_price'], 
    title=f"Profit Curve for {prod_info['name']}"
)
fig_profit.show()

## 5. Demand Curve
The fundamental inverse relationship between price and demand.

In [ ]:
fig_demand = plot_demand_curve(
    curve_data['prices'],
    curve_data['demands'],
    opt_result['optimal_price'],
    title=f"Demand Curve for {prod_info['name']}"
)
fig_demand.show()

## 6. Compare vs Current Price
How much better is the optimal price compared to the baseline?

In [ ]:
comp_result = optimizer.compare_with_current(prod_info['base_price'], base_features, competitor_price)
pd.DataFrame(comp_result).T

## 7. All Products Optimization
Run the optimization across the entire catalog.

In [ ]:
results = []
for pid, info in PRODUCT_CATALOG.items():
    opt = PriceOptimizer(demand_model, info['cost_price'], info['cost_price']*1.05, info['cost_price']*3)
    f = base_features.copy()
    f['product_id'] = pid
    comp_p = info['base_price']
    
    res = opt.optimize(f, comp_p)
    res['product_id'] = pid
    res['product_name'] = info['name']
    res['current_price'] = info['base_price']
    results.append(res)

results_df = pd.DataFrame(results)
results_df[['product_id', 'product_name', 'current_price', 'optimal_price', 'predicted_demand', 'daily_profit']]

## 8. Scenario Analysis
How should pricing change during weekends or festivals?

In [ ]:
scenario_engine = ScenarioEngine(optimizer)

base_scen = opt_result
fest_scen = scenario_engine.festival_scenario(base_features)
week_scen = scenario_engine.weekend_scenario(base_features)

scenarios = {
    'Baseline': base_scen,
    'Festival': fest_scen,
    'Weekend': week_scen
}

fig_scen = plot_scenario_comparison(scenarios, title="Optimal Price by Scenario")
fig_scen.show()

## 9. Sensitivity Analysis
How sensitive is our optimal price to competitor pricing?

In [ ]:
sensitivity_df = scenario_engine.sensitivity_analysis(base_features, competitor_price)
px.line(sensitivity_df, x='competitor_price', y='optimal_price', 
        title='Optimal Price vs. Competitor Price')

## 10. Key Insights
- The model balances volume and margin to find the peak profit.
- Optimal prices adapt to scenarios (e.g. higher demand during festivals allows for higher prices or higher volume through discounts).
- Competitor price has a direct impact on our optimal price due to the cross-price elasticity learned by the model.